In [1]:
from pathlib import Path
import sys

import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np


PACKAGE_ROOT = next((path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "interactions").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside AIJ_fig_submit.")
FIGURE_DIR = PACKAGE_ROOT / "Fig_07_noise"
ROOT = FIGURE_DIR / "source"
sys.path.insert(0, str(PACKAGE_ROOT / "shared"))

from tool import (  # noqa: E402
    apply_style,
    compute_newly_emerged_decomposition,
    compute_noise_interaction_distribution_fig3,
    list_harsanyi_samples,
    load_raw_interactions_for_samples,
    save_figure,
)


OUTPUT_DIR = FIGURE_DIR / "outputs"
CACHE_DIR = ROOT / "cache" / "generated"
INTERACTION_ROOT = PACKAGE_ROOT / "interactions" / "noise"
N_SHARED_SAMPLES = 50

ROWS = [
    {
        "label": "ResNet-20\nMNIST",
        "prefix": "1_cluster",
        "local_root": Path("dataset=mnist_model=resnet20_epoch=10_bs=128_lr=0.01_weight-decay=0.0005_seed=1029_noise-label=False"),
        "parameter_levels": [0.01, 0.02, 0.03, 0.04],
        "attack_levels": [1.0, 2.0, 3.0, 4.0],
    },
    {
        "label": "ResNet-20\nCIFAR-10",
        "prefix": "class-",
        "local_root": Path("dataset=cifar10_model=resnet20_epoch=50_bs=128_lr=0.01_weight-decay=0.0005_seed=1029_noise-label=False"),
        "parameter_levels": [0.01, 0.02, 0.03, 0.04],
        "attack_levels": [1.0, 2.0, 3.0, 4.0],
    },
    {
        "label": "VGG-11\nCIFAR-10",
        "prefix": "class-",
        "local_root": Path("dataset=cifar10_model=vgg11_epoch=50_bs=128_lr=0.01_weight-decay=0.0005_seed=1029_noise-label=False"),
        "parameter_levels": [0.01, 0.02, 0.03, 0.04],
        "attack_levels": [1.0, 2.0, 3.0, 4.0],
    },
]

N_PARAM_COLS = 4
N_COLS = 8
FONT_SIZE = 9
X_TICK_FONT_SIZE = 7
X_LABEL_FONT_SIZE = 13
BOTTOM_TITLE_FONT_SIZE = 14
LEGEND_FONT_SIZE = 14
FIGSIZE = (11.6, 4.55)

REAL_POS_COLOR = "#B84A62"
THEO_POS_COLOR = "#7A2634"
REAL_NEG_COLOR = "#3F6FA8"
THEO_NEG_COLOR = "#3F5F7F"
PANEL_FACE = "#FFFFFF"
SPINE_COLOR = "#C7CDD3"
ZERO_COLOR = "#AEB6BF"
ARROW_COLOR = "#5B9BD5"
BAR_ALPHA = 0.85
BAR_WIDTH = 0.7
THEORY_LINE_WIDTH = 1.1


def get_theoretical_spindle(decomposition):
    return compute_noise_interaction_distribution_fig3(
        decomposition["alpha"], decomposition["beta"], len(decomposition["noise_pos"])
    ).numpy()


def panel_abs_max(decomposition):
    real_pos = np.asarray(decomposition["noise_pos"])
    real_neg = np.asarray(decomposition["noise_neg"])
    fitted = get_theoretical_spindle(decomposition)
    return max(np.max(np.abs(real_pos)), np.max(np.abs(real_neg)), np.max(np.abs(fitted)))


def plot_fig3_style(ax, decomposition):
    real_pos = np.asarray(decomposition["noise_pos"])
    real_neg = np.asarray(decomposition["noise_neg"])
    fitted = get_theoretical_spindle(decomposition)

    x = np.arange(len(real_pos))
    ax.bar(x, real_pos, width=BAR_WIDTH, color=REAL_POS_COLOR, alpha=BAR_ALPHA, edgecolor="none", zorder=2)
    ax.bar(x, real_neg, width=BAR_WIDTH, color=REAL_NEG_COLOR, alpha=BAR_ALPHA, edgecolor="none", zorder=2)
    ax.plot(
        x, fitted, color=THEO_POS_COLOR, linewidth=THEORY_LINE_WIDTH, alpha=1.0,
        marker="o", markersize=2.2, markerfacecolor="white",
        markeredgecolor=THEO_POS_COLOR, markeredgewidth=0.8, zorder=4,
    )
    ax.plot(
        x, -fitted, color=THEO_NEG_COLOR, linewidth=THEORY_LINE_WIDTH, alpha=1.0,
        marker="o", markersize=2.2, markerfacecolor="white",
        markeredgecolor=THEO_NEG_COLOR, markeredgewidth=0.8, zorder=4,
    )

    ax.axhline(0, color=ZERO_COLOR, linewidth=0.45, linestyle="-", alpha=0.8, zorder=3)
    ax.set_xlim(-0.65, len(real_pos) - 0.35)
    ax.set_xticks([0, len(real_pos) - 1])
    ax.set_xticklabels(["1", str(len(real_pos))])
    ax.tick_params(axis="x", length=2.4, width=0.55, color="#4F4F4F", labelsize=X_TICK_FONT_SIZE, pad=1.5)
    ax.set_yticks([])
    ax.set_facecolor(PANEL_FACE)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(SPINE_COLOR)
        spine.set_linewidth(0.55)

    return panel_abs_max(decomposition)


def noisy_level_root(local_root, noise_kind, noise_value):
    if noise_kind == "parameter":
        return INTERACTION_ROOT / "parameter_gaussian" / Path(local_root) / f"noise_layer=CNN_gaussian_std={noise_value}", None
    if noise_kind == "attack":
        return INTERACTION_ROOT / "attack" / Path(local_root) / f"noise_lambda={noise_value}", f"attack_noise_{float(noise_value) / 255}"
    raise ValueError(f"Unknown noise_kind: {noise_kind}")


def sample_has_interactions(root, sample, nested=None):
    sample_dir = Path(root) / sample
    if nested is not None:
        sample_dir = sample_dir / nested
    return (sample_dir / "after_sparsify" / "Iand.npy").exists() and (
        sample_dir / "after_sparsify" / "Ior.npy"
    ).exists()


def get_row_shared_samples(config):
    local_root = Path(config["local_root"])
    clean_root = INTERACTION_ROOT / "clean" / local_root
    samples = []
    for sample in list_harsanyi_samples(clean_root, config["prefix"]):
        if not sample_has_interactions(clean_root, sample):
            continue
        roots = [
            noisy_level_root(local_root, "parameter", level)
            for level in config["parameter_levels"]
        ] + [
            noisy_level_root(local_root, "attack", level)
            for level in config["attack_levels"]
        ]
        if all(sample_has_interactions(root, sample, nested=nested) for root, nested in roots):
            samples.append(sample)

    if len(samples) < N_SHARED_SAMPLES:
        raise FileNotFoundError(
            f"Only {len(samples)} shared samples for {config['label']}; expected {N_SHARED_SAMPLES}"
        )
    return samples[:N_SHARED_SAMPLES]


def compute_shared_sample_decomposition(local_root, samples, noise_kind, noise_value):
    local_root = Path(local_root)
    clean_root = INTERACTION_ROOT / "clean" / local_root
    noisy_root, nested = noisy_level_root(local_root, noise_kind, noise_value)
    if not samples:
        raise FileNotFoundError(f"No shared samples for {noise_kind} level {noise_value}: {noisy_root}")

    clean_I_and, clean_I_or, masks = load_raw_interactions_for_samples(clean_root, samples)
    noisy_I_and, noisy_I_or, masks = load_raw_interactions_for_samples(noisy_root, samples, nested=nested)
    return compute_newly_emerged_decomposition(clean_I_and, clean_I_or, noisy_I_and, noisy_I_or, masks, tau=0.02)


def main():
    apply_style(FONT_SIZE)
    fig, axes = plt.subplots(len(ROWS), N_COLS, figsize=FIGSIZE)

    for r, config in enumerate(ROWS):
        samples = get_row_shared_samples(config)
        parameter_max = 0.0
        attack_max = 0.0

        for c, level in enumerate(config["parameter_levels"]):
            decomposition = compute_shared_sample_decomposition(
                config["local_root"],
                samples,
                "parameter",
                level,
            )
            parameter_max = max(parameter_max, plot_fig3_style(axes[r, c], decomposition))

        for c, level in enumerate(config["attack_levels"]):
            decomposition = compute_shared_sample_decomposition(
                config["local_root"],
                samples,
                "attack",
                level,
            )
            attack_max = max(attack_max, plot_fig3_style(axes[r, c + N_PARAM_COLS], decomposition))

        parameter_lim = max(parameter_max * 1.18, 1e-12)
        attack_lim = max(attack_max * 1.18, 1e-12)
        for c, ax in enumerate(axes[r]):
            group_lim = parameter_lim if c < N_PARAM_COLS else attack_lim
            ax.set_ylim(-group_lim, group_lim)
            ax.tick_params(axis="x", labelbottom=True)
            ax.set_xlabel(r"Order $m$", fontsize=X_LABEL_FONT_SIZE, labelpad=0)
            ax.xaxis.set_label_coords(0.5, -0.11)

    left_margin = 0.018
    right_margin = 0.99
    bottom_margin = 0.25
    top_margin = 0.982
    plt.subplots_adjust(
        left=left_margin,
        right=right_margin,
        bottom=bottom_margin,
        top=top_margin,
        wspace=0.055,
        hspace=0.35,
    )

    sep_x = (axes[0, N_PARAM_COLS - 1].get_position().x1 + axes[0, N_PARAM_COLS].get_position().x0) / 2
    fig.add_artist(
        Line2D(
            [sep_x, sep_x],
            [bottom_margin, top_margin],
            transform=fig.transFigure,
            color="#D2D8E0",
            linestyle="--",
            linewidth=1.0,
            alpha=1.0,
        )
    )

    fig.text(
        0.29,
        0.115,
        "Empirically measured newly emerged interaction distribution",
        ha="center",
        va="center",
        fontsize=BOTTOM_TITLE_FONT_SIZE,
        transform=fig.transFigure,
    )
    fig.text(
        0.77,
        0.115,
        "Theoretically derived spindle-shaped distribution",
        ha="center",
        va="center",
        fontsize=BOTTOM_TITLE_FONT_SIZE,
        transform=fig.transFigure,
    )

    def add_patch_item(x, y, color, label):
        fig.patches.append(
            mpatches.Rectangle(
                (x, y - 0.018),
                0.009,
                0.036,
                transform=fig.transFigure,
                facecolor=color,
                edgecolor="#222222",
                linewidth=0.7,
            )
        )
        fig.text(x + 0.04, y, label, ha="left", va="center", fontsize=LEGEND_FONT_SIZE, transform=fig.transFigure)

    def add_line_item(x, y, color, label):
        x0, x1 = x, x + 0.045
        xm = (x0 + x1) / 2
        fig.add_artist(
            Line2D(
                [x0, x1],
                [y, y],
                transform=fig.transFigure,
                color=color,
                linewidth=1.2,
            )
        )
        fig.add_artist(
            Line2D(
                [xm],
                [y],
                transform=fig.transFigure,
                color=color,
                marker="o",
                markersize=4.0,
                markerfacecolor="white",
                markeredgecolor=color,
                markeredgewidth=0.8,
                linestyle="None",
            )
        )
        fig.text(x + 0.055, y, label, ha="left", va="center", fontsize=LEGEND_FONT_SIZE, transform=fig.transFigure)

    add_patch_item(0.18, 0.055, REAL_POS_COLOR, r"$\Delta A^{(m),+}$")
    add_patch_item(0.34, 0.055, REAL_NEG_COLOR, r"$\Delta A^{(m),-}$")
    add_line_item(0.66, 0.055, THEO_POS_COLOR, r"$\mathcal{A}^{(m),+}_{\mathrm{spindle}}$")
    add_line_item(0.80, 0.055, THEO_NEG_COLOR, r"$\mathcal{A}^{(m),-}_{\mathrm{spindle}}$")

    save_figure(fig, OUTPUT_DIR, "main")
    plt.close(fig)


if __name__ == "__main__":
    main()


In [2]:
main()
